# Testing

In [1]:
import pandas as pd
from src.fine_tune_inference import FineTuneInference
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from pathlib import Path

In [2]:
fti = FineTuneInference(adapter_dir='/Users/micksmith/Neuromatic_Models/quarter-ft-v2')

In [11]:
def clean_messages(message: list):
    input_message = []
    gt_answer = None
    for m in message:
        if m['role'] == 'user' or m['role'] == 'system':
            input_message.append(m)
        elif m['role'] == 'assistant':
            gt_answer = m['content']
    return input_message, gt_answer

# Base Line Tests

In [20]:
class BaseTests:
    def __init__(self):
        has_cuda = torch.cuda.is_available()
        device = torch.device("cuda" if has_cuda else ("mps" if torch.backends.mps.is_available() else "cpu"))
        self.model_name = "Qwen/Qwen2.5-1.5B-Instruct"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            dtype=torch.bfloat16 if has_cuda else torch.float32,
            device_map=None
        ).to(device)
        self.max_tokens_dict = {
            'Sample Size': 8,
            'Quarter Variance': 256,
            'Sample Select': 2600
        }

    def generate_qwen_response(self, messages: list, model_task: str):
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_tokens_dict.get(model_task, 256),
            do_sample=False,
        )
        response = self.tokenizer.batch_decode(
            generated_ids[:, model_inputs.input_ids.shape[1]:],
            skip_special_tokens=True)[0]
        return response

    def base_test_eval(self, val_file_path: str, model_type: str):
        vdf = pd.read_json(val_file_path, lines=True, orient='records')
        test_messages = []
        actual = []
        for val in vdf.messages:
            imess, gt = clean_messages(val)
            test_messages.append(imess)
            actual.append(gt)
        predictions = [self.generate_qwen_response(x, model_type) for x in test_messages]
        eval_df = pd.DataFrame({
            'actual': actual,
            'predictions': predictions
        })
        print(f'Completed {model_type} Baseline Test')
        return eval_df



In [21]:
bt = BaseTests()


In [17]:
sample_size_base = bt.base_test_eval(
    '../src/models/Sample-Size-LLM/val.jsonl',
    'Sample Size'
)
quarter_var_base = bt.base_test_eval(
    '../src/models/Quarter-Variance-LLM/val.jsonl',
    'Quarter Variance'
)


Completed Sample Size Baseline Test
Completed Quarter Variance Baseline Test


In [23]:
sample_size_base.to_csv('../src/models/Sample-Size-LLM/sample_size_base.csv', index=False)
quarter_var_base.to_csv('../src/models/Quarter-Variance-LLM/quarter_var_base.csv', index=False)

In [22]:
sample_select_base = bt.base_test_eval(
    '../src/models/Sample-Select/val.jsonl',
    'Sample Select'
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

,actual,predictions
0,49,49
1,42,42
2,36,36
3,40,40
4,35,34
5,49,48
6,47,48
7,47,47
8,35,34
9,33,32


In [25]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", use_fast=True)


In [34]:
# max([len(tokenizer.encode(str(t))) for t in train_df.messages])

2521

In [8]:

new_task_message = [
    {
        "role": "system",
        "content": "You are an expert data analyst. Given summarized data from a tabular dataset, you will be asked to perform various statistical analyses. Return ONLY the quarter-on-quarter variance as a floating point number."},
    {
        "role": "user",
        "content": f"Quarterly Data: {qdf}\n\nReturn ONLY the quarter-on-quarter variance."}
]

In [9]:
fti.predict(new_task_message, 1000)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'1.51'

In [ ]:
# There is data in the attached file, <FILE>. I need to use the data to be able to accomplish <TASK> where the end result is <GT EXAMPLE>. Following the format of the data in the file, generate 400 synthetic examples of both the data in the file and a corresponding ground truth result.

In [33]:
sample_size_calculation1 = {
  "confidence_level": 0.9,
  "z_value": 1.64485,
  "tolerable_error": 0.1,
  "assumed_p": 0.5,
  "n0_unbounded": 67.63828806249998,
  "population_size": 203,
  "finite_population_corrected_n": 50.92219126352285,
  "final_sample_size": 51,
  "used_fpc": True,
  "rounding": "ceil",
  "formula_notes": {
    "n0": "(Z^2 * p * (1-p)) / E^2",
    "fpc": "(N * n0) / (N + n0 - 1)"
  }
}

In [35]:
population_size = sample_size_calculation1['population_size']
confidence_level = sample_size_calculation1['confidence_level']
tolerable_error = sample_size_calculation1['tolerable_error']
assumed_p = sample_size_calculation1['assumed_p']
ground_truth = sample_size_calculation1['final_sample_size']

In [36]:
sample_training_data = {
    'prompt': [
        {'content': "You are an expert data analyst. Given a tabular set of data, you will be asked to perform various statistical analyses.",
         'role': 'system'},
        {'content': f"There is a data set that has a population size of {population_size}. Calculate the required sample size for audit testing using the following statistical metrics:\n\n- Confidence level: {confidence_level}\n- Tolerable error rate: {tolerable_error}\n- Assumed probability of success: {assumed_p}\n\nReturn ONLY the final sample size.",
         'role': 'user'}
    ],
    'ground_truth': f"Final sample size: {ground_truth}"
}

# Sentence Transformers Eval

In [4]:
sample_size_eval = fti.score_predictions('/Users/micksmith/Library/CloudStorage/GoogleDrive-csmith715@gmail.com/My Drive/Neuromatic/Small_SLM_Task/sample_size_eval.csv')
quarter_var_eval = fti.score_predictions('/Users/micksmith/Library/CloudStorage/GoogleDrive-csmith715@gmail.com/My Drive/Neuromatic/Small_SLM_Task/quarter_variance_eval.csv')
sample_select_eval = fti.score_predictions('/Users/micksmith/Library/CloudStorage/GoogleDrive-csmith715@gmail.com/My Drive/Neuromatic/Small_SLM_Task/sample_select_eval.csv')

In [6]:
print(f'Sample Size Score: {sample_size_eval}\nQuarter Variance Score: {quarter_var_eval}\nSample Select Score: {sample_select_eval}')

Sample Size Score: 0.9577656984329224
Quarter Variance Score: 0.959423840045929
Sample Select Score: 0.9998720288276672


In [25]:
sample_size_base.to_csv('../src/models/Sample-Size-LLM/sample_size_base.csv', index=False)
quarter_var_base.to_csv('../src/models/Quarter-Variance-LLM/quarter_var_base.csv', index=False)

bss = fti.score_predictions('../src/models/Sample-Size-LLM/sample_size_base.csv')
qvs = fti.score_predictions('../src/models/Quarter-Variance-LLM/quarter_var_base.csv')

In [2]:
file_path = Path('../src/models/Quarter-Variance-LLM/quarter_var_base.csv')

In [10]:
file_path.stem

'quarter_var_base'

In [27]:
sss = fti.score_predictions('/Users/micksmith/Library/CloudStorage/GoogleDrive-csmith715@gmail.com/My Drive/Neuromatic/Small_SLM_Task/sample_select_base.csv')

In [30]:
print(f'Qwen/Qwen2.5-1.5B-Instruct Baseline Evaluation Results\n\nSample Size Base Score: {bss}\nQuarter Variance Base Score: {qvs}\nSample Select Base Score: {sss}')

Qwen/Qwen2.5-1.5B-Instruct Baseline Evaluation Results

Sample Size Base Score: 0.6554078459739685
Quarter Variance Base Score: 0.2585287094116211
Sample Select Base Score: 0.46143126487731934


In [3]:
a = fti.score_predictions('../src/models/Sample-Size-LLM/sample_size_base.csv')
b = fti.score_predictions('../src/models/Quarter-Variance-LLM/quarter_var_base.csv')
c = fti.score_predictions('/Users/micksmith/Library/CloudStorage/GoogleDrive-csmith715@gmail.com/My Drive/Neuromatic/Small_SLM_Task/sample_select_base.csv')

In [12]:
ssdf = pd.read_csv('/Users/micksmith/Library/CloudStorage/GoogleDrive-csmith715@gmail.com/My Drive/Neuromatic/Small_SLM_Task/sample_size_eval.csv').astype(str)

In [10]:
st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [13]:
emb1 = st_model.encode(ssdf['actual'].tolist(), convert_to_tensor=True, normalize_embeddings=True)
emb2 = st_model.encode(ssdf['predictions'].tolist(), convert_to_tensor=True, normalize_embeddings=True)
sims = util.pairwise_cos_sim(emb1, emb2)

KeyError: 'predictions'